# Sales by region, with Polars

A small sales log, net of returns: which region earned most, and what did each month bring in?

This notebook needs the **Polars kernel**: rnx with the Polars adapter assembled in (see the README's
"Polars notebook" section). The data is the bundled `demos/data/sales.csv`; the kernel runs in this
notebook's directory, so the path is `../data/sales.csv`. Prices and revenues are integer cents; the mean order is reported as a float.

Two helpers: cents as dollars, and one integer read out of a frame.

In [1]:
fn dollars(cents) { format!("{}.{:02}", cents / 100, cents % 100) }

fn int_at(frame, name, row) { Ok(frame.column(name)?.i64()?.get(row)?.unwrap()) }

Load the CSV with an explicit schema. A frame left as a cell's value shows a bounded preview:
the first 10 rows and at most 8 columns, with a line saying what was left out.

In [2]:
let sales = polars::read_csv("../data/sales.csv", [("month", "string"), ("region", "string"),
	("product", "string"), ("units", "i64"), ("price_cents", "i64"), ("returned", "bool")])?;
sales

DataFrame: 24 rows × 6 columns
[14 rows and 0 columns omitted by display limits]
"month": string | "region": string | "product": string | "units": i64 | "price_cents": i64 | "returned": bool
"2026-07" | "north" | "widget" | 3 | 1250 | false
"2026-07" | "south" | "gadget" | 1 | 4000 | false
"2026-07" | "east" | "widget" | 2 | 1250 | true
"2026-07" | "west" | "gizmo" | 4 | 2500 | false
"2026-07" | "north" | "gizmo" | 1 | 2500 | false
"2026-07" | "east" | "gadget" | 2 | 4000 | false
"2026-07" | "south" | "widget" | 6 | 1250 | false
"2026-07" | "west" | "widget" | 2 | 1250 | false
"2026-08" | "north" | "gadget" | 2 | 4000 | false
"2026-08" | "west" | "widget" | 5 | 1250 | false


Leave out returned orders and derive each order's revenue. `not` is a Rune keyword, so the
binding is `not_()`; column arithmetic can fail, hence the `?` before `.alias`.

In [3]:
let revenue = (polars::col("units") * polars::col("price_cents"))?.alias("revenue_cents");
let kept = sales.lazy().filter(polars::col("returned").not_()).with_columns([revenue])?.collect()?;
kept

DataFrame: 20 rows × 7 columns
[10 rows and 0 columns omitted by display limits]
"month": string | "region": string | "product": string | "units": i64 | "price_cents": i64 | "returned": bool | "revenue_cents": i64
"2026-07" | "north" | "widget" | 3 | 1250 | false | 3750
"2026-07" | "south" | "gadget" | 1 | 4000 | false | 4000
"2026-07" | "west" | "gizmo" | 4 | 2500 | false | 10000
"2026-07" | "north" | "gizmo" | 1 | 2500 | false | 2500
"2026-07" | "east" | "gadget" | 2 | 4000 | false | 8000
"2026-07" | "south" | "widget" | 6 | 1250 | false | 7500
"2026-07" | "west" | "widget" | 2 | 1250 | false | 2500
"2026-08" | "north" | "gadget" | 2 | 4000 | false | 8000
"2026-08" | "west" | "widget" | 5 | 1250 | false | 6250
"2026-08" | "east" | "gizmo" | 2 | 2500 | false | 5000


Revenue by region: units, revenue and order count. Group order isn't stable, so the result is
sorted, largest revenue first.

In [4]:
let by_region = kept.lazy().group_by([polars::col("region")])?
	.agg([polars::col("units").sum().alias("units"),
		polars::col("revenue_cents").sum().alias("revenue_cents"),
		polars::len().alias("orders")])?
	.sort_by_exprs([polars::col("revenue_cents")],
		polars::SortMultipleOptions::new().with_order_descending(true))?
	.collect()?;
by_region

DataFrame: 4 rows × 4 columns
"region": string | "units": i64 | "revenue_cents": i64 | "orders": u32
"north" | 16 | 38750 | 6
"west" | 19 | 31250 | 5
"south" | 14 | 27000 | 5
"east" | 12 | 25750 | 4


Revenue by month, in month order.

In [5]:
let by_month = kept.lazy().group_by([polars::col("month")])?
	.agg([polars::col("revenue_cents").sum().alias("revenue_cents")])?
	.sort_by_exprs([polars::col("month")], polars::SortMultipleOptions::new())?
	.collect()?;
by_month

DataFrame: 3 rows × 2 columns
"month": string | "revenue_cents": i64
"2026-07" | 38250
"2026-08" | 41000
"2026-09" | 43500


The total and the mean order, in cents. `select` is also a Rune keyword: the binding is
`select_()`. The mean is a float.

In [6]:
let summary = kept.lazy().select_([polars::col("revenue_cents").sum().alias("total_cents"),
	polars::col("revenue_cents").mean().alias("mean_cents")])?.collect()?;
summary

DataFrame: 1 rows × 2 columns
"total_cents": i64 | "mean_cents": f64
122750 | 6137.5


The answer, read back out of the frames. The top region's share is in tenths of a percent,
rounded half up, in integers.

In [7]:
let orders = kept.height()?;
let returned = sales.height()? - orders;
let total = int_at(summary, "total_cents", 0)?;
let top = by_region.column("region")?.str()?.get(0)?.unwrap();
let top_cents = int_at(by_region, "revenue_cents", 0)?;
let tenths = (top_cents * 2000 + total) / (2 * total);
println!("Net revenue: ${} from {} orders ({} returned, left out)", dollars(total), orders, returned);
println!("Top region: {}, ${}, {}.{}% of net revenue", top, dollars(top_cents), tenths / 10, tenths % 10);

Net revenue: $1227.50 from 20 orders (4 returned, left out)
Top region: north, $387.50, 31.6% of net revenue
